# Test: test-time fine-tuning on VOT-LT2020 and GOT-10k

Runs STARK-ST with or without test-time fine-tuning of the classification head on VOT-LT2020 (vot-toolkit) or a
GOT-10k split and evaluates it (COCO-style metrics; GOT-10k: also AO / SR). Details: [docs/test.md](../docs/test.md)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # Colab only: get / update the code
    os.system("git clone -q https://github.com/balk21/stark-cls-finetune /content/stark-cls-finetune || git -C /content/stark-cls-finetune pull -q")
    os.chdir("/content/stark-cls-finetune/notebooks")
sys.path.insert(0, os.getcwd())
import importlib, nbhelper; nb = importlib.reload(nbhelper)
nb.init(colab_drive="/content/drive/MyDrive/LOKAP")  # finds / creates the vot1 environment; colab_drive: Drive folder (Colab only)

## Weights

In [ ]:
nb.list_weights()                    # official STARK weights and the runs trained here (by origin): values for `weights`
# nb.cli("smoke", "--frames", "50")  # optional quick check without vot-toolkit (downloads ballet, 57 MB, if needed)

## Parameters

In [ ]:
PARAMS = dict(
    name=None,                     # output folder <outputs>/<name>; None = generated from the parameters
    model="stark_st",              # "stark_st" (STARK-ST) | "stark_s" (STARK-S50: no confidence score, ft_mode must be "none")
    model_config="baseline_R101",  # stark_st: "baseline_R101" (ST101) | "baseline" (ST50); stark_s: "baseline" (S50); + "_got10k_only": trained on GOT-10k only
    weights="official",            # "official": STARK's weights (downloaded when needed) | "<run name>" from nb.list_weights() | checkpoint path
    dataset="votlt2020",           # "votlt2020" (vot-toolkit) | "got10k_val" | "got10k_test" (zip for the GOT-10k server) | "got10k_train"
    sequences=["bull"],            # "all" | VOT: ["bull", ...] (downloaded automatically) | GOT-10k: ["GOT-10k_Val_000001", ...]
    update_mode="stark",           # "stark": every N-th frame becomes the template | "max": the best-scoring frame of every N frames
    update_interval=100,           # N (STARK: 100 on VOT-LT, 200 on GOT-10k); 99999 = no updates
    update_conf_thr=0.5,           # update only if the score is above this
    update_iou_thr=0.5,            # "max" only: min IoU with the previous candidate frame's box; 0 = no IoU check
    max_template_updates=-1,       # per sequence; -1 = unlimited
    ft_mode="none",                # "none": plain STARK-ST | "init": first frame | "online": first frame + every template update
    ft_samples="pos",              # "pos": positive sample only | "posneg": + a negative region of the same frame
    ft_lr=1e-4,                    # fine-tuning learning rate (AdamW)
    ft_epochs_init=15,             # steps on the first frame
    ft_epochs_online=1,            # steps per online fine-tuning
    ft_weight_decay=1e-4,
    ft_grad_clip_norm=0.1,
    ft_pos_jitter=True,            # True: jitter the positive box as in STARK stage-2 training | False: exact box
    ft_center_jitter=4.5,          # jitter strength (STARK values)
    ft_scale_jitter=0.5,
    max_ft_updates=-1,             # online fine-tunings per sequence; -1 = unlimited
    seed=0,                        # same seed = same result (on the same GPU type)
    eval_score_thr=0.35,           # score threshold: mAP / AP50 / AP75 and P / R / F1 count the predictions above it
    eval_iou_thr=0.5,              # IoU of a correct detection (P / R / F1)
    eval_thr_resolution=100,       # candidate thresholds in the F-max search (vot-toolkit: 100)
    run_redetection=False,         # True: also the VOT-LT "redetection" experiment (~2x time)
    tracker_timeout=300,           # seconds per tracker response
)
nb.describe(PARAMS)  # checks the parameters; shows output folder, weights, sequences to download

## Run

In [ ]:
OUT = nb.test(PARAMS)  # same name + parameters again: resumes (skips finished sequences) | overwrite=True: starts again

## Results

In [ ]:
nb.show_summary(OUT)                      # mAP / AP50 / AP75, F-max threshold, fixed threshold, per sequence
nb.show_f_curve(OUT)                      # P / R / F vs. threshold and the PR curve
nb.show_plots(OUT)                        # IoU and score vs. frame, fine-tuning loss | sequences=["bull"]: only these

## Optional

In [ ]:
# nb.analyze(OUT, score_thr=0.5, iou_thr=0.5)                             # metrics again with other evaluation settings (no tracking)
# nb.test_many([dict(PARAMS, ft_mode=m) for m in ("none", "init", "online")])  # several experiments one after another
# nb.test_many([dict(PARAMS, weights=w) for w in ("official", "st101_coco_stage2")])  # the same test with other weights
# nb.test_many([dict(PARAMS, update_mode=m) for m in ("stark", "max")])  # the two template update rules
# MODELS = [("stark_st", "baseline_R101"), ("stark_st", "baseline"), ("stark_s", "baseline")]  # ST101, ST50, S50
# nb.test_many([dict(PARAMS, model=m, model_config=c, ft_mode="none") for m, c in MODELS])  # official weights: downloaded when needed
# STARK's GOT-10k-only weights on all GOT-10k val sequences / on the GOT-10k test split (zip for the server) / on VOT:
# G = dict(PARAMS, model_config="baseline_R101_got10k_only", ft_mode="none", update_interval=200, sequences="all")
# nb.test_many([dict(G, dataset="got10k_val"), dict(G, dataset="got10k_test"), dict(G, dataset="votlt2020", update_interval=100)])